In [1]:
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.naive import NaiveForecaster

warnings.filterwarnings("ignore")

In [2]:
def smape(y_true: pd.Series, y_pred: list, decimals: int = 2) -> float:
  y_true = np.array(y_true)
  y_pred = np.array(y_pred)
  mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
  y_true = y_true[mask]
  y_pred = y_pred[mask]
  numerator = np.abs(y_true - y_pred)
  denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
  epsilon = 1e-10
  result = np.mean(numerator / (denominator + epsilon)) * 100
  return round(result, decimals)

In [3]:
def evaluate_dataset(csv_path: str):
  df = pd.read_csv(csv_path)
  df["Date"] = pd.to_datetime(df["Date"])
  df = df.sort_values("Date")
  y = df.set_index("Date")["Revenue"].astype(float)
  y.index.freq = pd.infer_freq(y.index)

  cutoff = "2011-08-31"
  y_train = y.loc[:cutoff]
  y_test  = y.loc["2011-09-01":]

  fh = ForecastingHorizon(np.arange(1, len(y_test) + 1), is_relative=True)

  print("=" * 80)
  print(csv_path)
  print(f"TRAIN: {len(y_train)} | TEST: {len(y_test)}")
  print("=" * 80)

  models = {
    "Naive":         NaiveForecaster(strategy="last"),
    "SeasonalNaive": NaiveForecaster(strategy="last", sp=7),
  }

  results = {}
  for name, model in models.items():
    print(f"\nFitting {name}...")
    model.fit(y_train)
    pred = model.predict(fh)
    pred_values = pred.values
    results[name] = {
      "pred":  pred_values.tolist(),
      "mae":   round(mean_absolute_error(y_test, pred_values), 2),
      "rmse":  round(np.sqrt(mean_squared_error(y_test, pred_values)), 2),
      "smape": smape(y_test, pred_values),
    }
    print(f"  MAE={results[name]['mae']:.4f}  RMSE={results[name]['rmse']:.4f}  SMAPE={results[name]['smape']:.2f}%")

  return {
    "dataset": csv_path,
    "n_pred":  len(y_test),
    "models":  results,
  }

In [4]:
import os
os.makedirs("resultados", exist_ok=True)

DATASETS = ["product.csv", "country.csv", "customer.csv"]

serie_map = {"product": "Produto", "country": "País", "customer": "Cliente"}
rows = []

for dataset in DATASETS:
  res        = evaluate_dataset("data/" + dataset)
  serie_name = serie_map[dataset.replace(".csv", "")]

  for modelo, metrics in res["models"].items():
    rows.append({
      "serie":              serie_name,
      "periodos_previstos": res["n_pred"],
      "modelo":             modelo,
      "valores_previstos":  metrics["pred"],
      "MAE":                metrics["mae"],
      "RMSE":               metrics["rmse"],
      "SMAPE":              metrics["smape"],
    })

df_results = pd.DataFrame(rows)
df_results.to_csv("resultados/resultados_naive.csv", index=False)
print("Resultados salvos em resultados/resultados_naive.csv")
print(df_results[["serie", "modelo", "MAE", "RMSE", "SMAPE"]])

data/product.csv
TRAIN: 383 | TEST: 71

Fitting Naive...
  MAE=126.4800  RMSE=152.7600  SMAPE=111.84%

Fitting SeasonalNaive...
  MAE=127.0200  RMSE=151.5700  SMAPE=118.19%
data/country.csv
TRAIN: 457 | TEST: 72

Fitting Naive...
  MAE=14876.2800  RMSE=16097.3900  SMAPE=98.74%

Fitting SeasonalNaive...
  MAE=13028.7800  RMSE=14979.6600  SMAPE=89.17%
data/customer.csv
TRAIN: 639 | TEST: 99

Fitting Naive...
  MAE=529.0800  RMSE=911.4000  SMAPE=105.05%

Fitting SeasonalNaive...
  MAE=595.8900  RMSE=936.6200  SMAPE=107.19%
Resultados salvos em resultados/resultados_naive.csv
     serie         modelo       MAE      RMSE   SMAPE
0  Produto          Naive    126.48    152.76  111.84
1  Produto  SeasonalNaive    127.02    151.57  118.19
2     País          Naive  14876.28  16097.39   98.74
3     País  SeasonalNaive  13028.78  14979.66   89.17
4  Cliente          Naive    529.08    911.40  105.05
5  Cliente  SeasonalNaive    595.89    936.62  107.19


/home/wesley/Work/engine/ENIAC_2026/.venv/lib/python3.12/site-packages/sktime/utils/seasonality.py:174: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  df.index = df.index.to_period(freq=freq)
/home/wesley/Work/engine/ENIAC_2026/.venv/lib/python3.12/site-packages/sktime/utils/seasonality.py:119: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  aix = anchor.index.to_period(freq=freq)
/home/wesley/Work/engine/ENIAC_2026/.venv/lib/python3.12/site-packages/sktime/utils/seasonality.py:162: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  df_pivot.index = df_pivot.index.to_timestamp(
/home/wesley/Work/engine/ENIAC_2026/.venv/lib/python3.12/site-packages/sktime/utils/seasonality.py:174: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a f